# base Module

Contains core functionality used by both the Gaussian Process and Neural Network methods

## Imports

In [ ]:
import numpy as np
import pandas as pd
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable, Tuple, Any
from pathlib import Path
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler

## Data Classes

Defines some basic data structures

In [ ]:
@dataclass(frozen=True)
class Prediction:
    mean: np.ndarray
    std: np.ndarray

@dataclass
class OptimizationResult:
    next_coords: np.ndarray
    score: float
    acquisition_name: str
    model_name: str
    predicted_mean: float
    predicted_std: float

## Acquisition Functions

Defines the acquisition strategies for UCB and EI

In [ ]:
class AcquisitionStrategy(ABC):
    @property
    @abstractmethod
    def name(self) -> str:
        pass

    @abstractmethod
    def score(self, prediction: Prediction, current_best_y: float) -> np.ndarray:
        pass

class UCB(AcquisitionStrategy):
    def __init__(self, kappa: float = 1.96):
        self.kappa = kappa
    
    @property
    def name(self) -> str:
        return "UCB"

    def score(self, prediction: Prediction, current_best_y: float) -> np.ndarray:
        return prediction.mean + (self.kappa * prediction.std)

class ExpectedImprovement(AcquisitionStrategy):
    def __init__(self, xi: float = 0.01):
        self.xi = xi

    @property
    def name(self) -> str:
        return "EI"

    def score(self, prediction: Prediction, current_best_y: float) -> np.ndarray:
        std = np.maximum(prediction.std, 1e-9)
        z = (prediction.mean - current_best_y - self.xi) / std
        return (prediction.mean - current_best_y - self.xi) * norm.cdf(z) + std * norm.pdf(z)

## BaseBayesianOptimizer

Defines the parent class used by both the GP and NN BO classes. If applicable, applies a transformation data based on the experiment configuration.

### predict

Method used by the plotter module. The @abstractmethod _predict_normalized requires the subclass to have a prediction method. The predict method calls this returns either the scaled mean and standard deviation or performs a reverse transformation. 

### _calc_score

Method used by the plotter module. Returns the result of the acquisition function for given x.

### suggest

Method used by the plotter module. Returns an OptimizationResult instance for each model with the suggested next point and the predicted mean and standard deviation in the original scale.

In [2]:
class BaseBayesianOptimizer(ABC):
    def __init__(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        bounds: Tuple[float, float] = (0.0, 1.0),
        symlog_transform_y: bool = False,
        log_transform_y: bool = False,
        seed: int = 42,
        history_path: Any = None
    ):
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.X = X_train
        self.raw_y = y_train
        self.bounds = bounds
        self.n_dims = X_train.shape[1]
        self.symlog_transform_y = symlog_transform_y
        self.log_transform_y = log_transform_y
        self.history = pd.read_csv(history_path) if history_path and Path(history_path).exists() else None

        self.scaler = StandardScaler()
        if self.symlog_transform_y:
            y_proc = np.sign(y_train) * np.log1p(np.abs(y_train))
        if self.log_transform_y:
            if np.any(y_train <= 0):
                print("Warning: y contains non-positive values. Log transform may fail.")
            y_proc = np.log(y_train)
        else:
            y_proc = y_train
        
        self.y_norm = self.scaler.fit_transform(y_proc.reshape(-1, 1)).flatten()

            @abstractmethod
        
    def _predict_normalized(self, model: Any, X: np.ndarray) -> Prediction:
        pass

    def predict(self, model: Any, X: np.ndarray, return_std: bool = False, return_scaled: bool = False):
        norm_pred = self._predict_normalized(model, X)
        
        if return_scaled:
            if return_std: return norm_pred.mean, norm_pred.std
            return norm_pred.mean

        mu_trans = self.scaler.inverse_transform(norm_pred.mean.reshape(-1, 1)).flatten()
        std_trans = norm_pred.std * self.scaler.scale_[0]
        if self.symlog_transform_y:
            mu_orig = np.sign(mu_trans) * (np.expm1(np.abs(mu_trans)))
            if return_std:
                std_orig = mu_orig * std_trans 
                return mu_orig, std_orig
            return mu_orig
        if self.log_transform_y:
            mu_orig = np.exp(mu_trans)
            if return_std:
                std_orig = mu_orig * std_trans 
                return mu_orig, std_orig
            return mu_orig
        else:
            if return_std: return mu_trans, std_trans
            return mu_trans

    def _calc_score(self, X_candidates, model, acquisition: AcquisitionStrategy, repulsion: float = 0.0):
        pred = self._predict_normalized(model, X_candidates)
        scores = acquisition.score(pred, current_best_y=self.y_norm.max())
       
        return scores

    def suggest(self, model, acquisition: AcquisitionStrategy, optimizer_func: Callable, **kwargs):
        def objective(x_flat):
            x = x_flat.reshape(1, -1)
            s = self._calc_score(x, model, acquisition, **kwargs)
            return -float(s[0])

        best_x, _ = optimizer_func(objective, self.bounds, self.n_dims)

        mu_orig, std_orig = self.predict(model, [best_x], return_std=True)
        raw_score = self._calc_score(best_x.reshape(1,-1), model, acquisition, **kwargs)[0]
        
        final_score_display = raw_score * self.scaler.scale_[0] 

        return OptimizationResult(
            next_coords=best_x,
            score=final_score_display,
            acquisition_name=acquisition.name,
            model_name="Model",
            predicted_mean=mu_orig[0],
            predicted_std=std_orig[0]
        )

NameError: name 'ABC' is not defined